# 01d - Grad-CAM: O Que o Modelo Está Detectando?

Grad-CAM (Gradient-weighted Class Activation Mapping) visualiza quais regiões da imagem mais influenciam a decisão do modelo.

**Objetivo:** determinar se o modelo aprende features semânticas de rostos (olhos, pele, estrutura facial) ou artefatos triviais (bordas de blocos JPEG, padrões de textura do StyleGAN).

> Requer o modelo treinado pelo notebook `03_treinamento_resnet50.ipynb`.

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import cv2
from PIL import Image
from torchvision import models, transforms

In [ ]:
PROJECT_ROOT    = Path.cwd().resolve().parent
_data_root_file = PROJECT_ROOT / "data_root.env"
DATA_ROOT       = Path(_data_root_file.read_text().strip()) if _data_root_file.exists() else PROJECT_ROOT / "data"

RAW_DIR     = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
MODEL_DIR   = PROJECT_ROOT / "artifacts" / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# detecta o modelo ResNet-50 mais recente (qualquer preprocessing)
model_files = sorted(MODEL_DIR.glob("resnet50_*.pth"), key=lambda p: p.stat().st_mtime)
# ignora arquivos de history/metrics que também começam com resnet50_
model_files = [p for p in model_files if "history" not in p.name and "metrics" not in p.name]
if not model_files:
    raise FileNotFoundError("Nenhum modelo ResNet-50 encontrado. Execute o notebook 03 primeiro.")
MODEL_PATH = model_files[-1]

print("Device:", DEVICE)
print("Modelo:", MODEL_PATH)
print("Dados:", RAW_DIR)

## 1. Carrega o Modelo

In [ ]:
def build_resnet50(num_classes=2, dropout=0.5):
    model = models.resnet50(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, num_classes)
    )
    return model

model = build_resnet50().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

CLASSES = ["fake", "real"]
print("Modelo carregado:", MODEL_PATH.name)

## 2. Implementação do Grad-CAM

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.activations = None
        self.gradients   = None
        target_layer.register_forward_hook(self._save_activations)
        target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        output = model(input_tensor)
        pred_class = output.argmax(dim=1).item()
        if class_idx is None:
            class_idx = pred_class
        model.zero_grad()
        output[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, pred_class, float(torch.softmax(output, dim=1)[0, pred_class])

gradcam = GradCAM(model, model.layer4[-1])
print("Grad-CAM inicializado na camada: model.layer4[-1]")

## 3. Transforms e Carregamento

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def load_sample(folder, n=8, seed=42):
    paths = sorted(folder.glob("*.jpg"))
    return random.Random(seed).sample(paths, min(n, len(paths)))

def apply_gradcam(img_path, class_idx=None):
    img_orig = Image.open(img_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    tensor = transform(img_orig).unsqueeze(0).to(DEVICE)
    cam, pred, conf = gradcam.generate(tensor, class_idx)
    cam_resized = cv2.resize(cam, (IMAGE_SIZE, IMAGE_SIZE))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = np.array(img_orig) * 0.5 + heatmap * 0.5
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    return img_orig, overlay, CLASSES[pred], conf

real_paths = load_sample(RAW_DIR / "test" / "real", n=8)
fake_paths = load_sample(RAW_DIR / "test" / "fake", n=8)
print(f"Amostras: {len(real_paths)} reais, {len(fake_paths)} falsas")

## 4. Grad-CAM — Imagens Reais

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(20, 6))

for col, path in enumerate(real_paths):
    img, overlay, pred, conf = apply_gradcam(path)
    axes[0, col].imshow(img)
    axes[0, col].set_title("original", fontsize=7)
    axes[0, col].axis("off")
    axes[1, col].imshow(overlay)
    axes[1, col].set_title(f"{pred} {conf:.2f}", fontsize=7,
                           color="green" if pred == "real" else "red")
    axes[1, col].axis("off")

plt.suptitle("Grad-CAM — Imagens Reais", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "gradcam_real.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Grad-CAM — Imagens Falsas (StyleGAN)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(20, 6))

for col, path in enumerate(fake_paths):
    img, overlay, pred, conf = apply_gradcam(path)
    axes[0, col].imshow(img)
    axes[0, col].set_title("original", fontsize=7)
    axes[0, col].axis("off")
    axes[1, col].imshow(overlay)
    axes[1, col].set_title(f"{pred} {conf:.2f}", fontsize=7,
                           color="green" if pred == "fake" else "red")
    axes[1, col].axis("off")

plt.suptitle("Grad-CAM — Imagens Falsas (StyleGAN)", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "gradcam_fake.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Espectro 2D — Confirma Artefatos JPEG

O espectro 2D (não radial) mostra padrões direcionais — o grid 8x8 do JPEG aparece como pontos brilhantes em frequências específicas.

In [ ]:
def mean_fft_2d(paths, size=224, n=200, seed=42):
    paths = random.Random(seed).sample(paths, min(n, len(paths)))
    spectra = []
    for p in paths:
        img = np.array(Image.open(p).convert("L").resize((size, size)), dtype=np.float32) / 255.0
        f = np.fft.fftshift(np.fft.fft2(img))
        spectra.append(np.log1p(np.abs(f)))
    return np.mean(spectra, axis=0)

real_fft2d = mean_fft_2d(real_paths + load_sample(RAW_DIR / "test" / "real", n=200))
fake_fft2d = mean_fft_2d(fake_paths + load_sample(RAW_DIR / "test" / "fake", n=200))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
vmin = min(real_fft2d.min(), fake_fft2d.min())
vmax = max(real_fft2d.max(), fake_fft2d.max())

axes[0].imshow(real_fft2d, cmap="magma", vmin=vmin, vmax=vmax)
axes[0].set_title("Espectro 2D — Real")
axes[0].axis("off")

axes[1].imshow(fake_fft2d, cmap="magma", vmin=vmin, vmax=vmax)
axes[1].set_title("Espectro 2D — Fake (StyleGAN)")
axes[1].axis("off")

diff = fake_fft2d - real_fft2d
im = axes[2].imshow(diff, cmap="RdBu_r", vmin=-0.5, vmax=0.5)
axes[2].set_title("Diferença (Fake − Real)")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.suptitle("Espectro 2D: identificação de artefatos direcionais", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "espectro_2d_artefatos.png", dpi=150, bbox_inches="tight")
plt.show()